# Arc Lamps

Everything so far has worked in pixels. Chapter 4 found *where* on the detector the source
falls, but nothing in the science frame says which **wavelength** any column holds, and the
instrument cannot tell us either: a grism disperses light by an amount that depends on
alignment, temperature and exactly where the slit sits, so the mapping from column to
wavelength has to be measured for the observation in hand.

The standard way to measure it is to expose the detector to a gas-discharge lamp. Each line
such a lamp emits is a monochromatic image of the slit at a laboratory-known wavelength, so
the arc frame carries a ruler laid directly across the detector. Lamps are not the only ruler.
The night-sky emission lines, mostly OH airglow in the red and near-infrared, are recorded in
every science exposure through the same optics and at the same time, which makes them the
calibrator of choice when the instrument flexes between the arc and the science exposure or
when no lamp exposure exists. Telluric absorption bands serve the same purpose against a
bright continuum, and stabilized high-resolution spectrographs use Fabry-Pérot etalons or
laser frequency combs for a dense, evenly spaced set of reference lines. Whatever the source,
the requirement is the same: features at known wavelengths spread across the detector.

This chapter is about **the arc frames as data**: what the dataset ships, how the arc
exposures differ from the science exposure, and the one property of these frames that shapes
the next three chapters. The book calibrates with the lamps, but nothing downstream depends on
that choice: `specreduce` needs a spectrum with lines in it and a list of the wavelengths those
lines should have, and a sky spectrum paired with an airglow line list would run through the
same steps. Detecting the lines and working out which catalog wavelength each one is belongs
to [Chapter 9](09-line-identification.ipynb); nothing here needs to know a single wavelength.

In [ ]:
import sys

sys.path.append("../src")

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits as pf
from astropy.visualization import simple_norm

from common import ARC_LAMPS, DATA_DIR, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
nx = obj.data.shape[1]

## What the dataset ships

Three arc exposures, one per lamp (HgAr, Ne and Xe), and the science frame. Their headers
record how each was taken:

In [ ]:
files = {lamp: f"osiris_arc_{lamp}.fits.bz2" for lamp in ARC_LAMPS}
files["TrES-3b"] = "osiris_tres_3b.fits.bz2"

print(f"{'frame':>9}  {'exptime':>7}  {'grism':>7}  {'slit':>21}  {'observed':>20}")
for label, name in files.items():
    header = pf.getheader(DATA_DIR / name)
    print(
        f"{label:>9}  {header['EXPTIME']:6.1f}s  {header['GRISM']:>7}  "
        f"{header['MASKNAME']:>21}  {header['DATE-OBS'][:19]:>20}"
    )

Three things in that table matter later.

**The grism is the same.** All four frames were taken through `R1000R`, so the arc frames
disperse light exactly as the science frame does. An arc taken through a different grism would
be useless as a wavelength reference.

**The exposure times differ**, at 6.2, 2.3 and 8.0 seconds against the science frame's 12, and
they are chosen per lamp to bring the brightest lines up without saturating them. The raw peaks
below land within about a quarter of each other, which is the point: per second of exposure Ne
is some three times brighter than the other two, and its exposure is shortened to compensate.

**The arcs were taken on a different night.** `DATE-OBS` records UT, so read the dates as
observing nights rather than calendar dates: the science frame was exposed after midnight and
belongs to the night of 2014-07-07 even though its `DATE-OBS` says the 8th, and all three arcs
are from the evening of 2014-07-09, about 1.6 days later. Calibration frames taken two nights
from the science exposure are a compromise worth being aware of. A spectrograph can flex
between nights, and the wavelength solution derived here inherits whatever it did. Nothing in
this dataset lets us check that, so the honest position is to note it and move on.

The frames themselves, on the same logarithmic stretch as Chapter 1:

In [ ]:
fig, axs = plt.subplots(3, figsize=(8, 5.4), sharex="all", constrained_layout=True)

for ax, frame, lamp in zip(axs, arcs, lamps):
    ax.imshow(
        frame.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(frame.data, stretch="log", vmin=0, vmax=250_000),
    )
    ax.text(0.01, 0.92, lamp, va="top", ha="left", transform=ax.transAxes)

plt.setp(axs, ylabel="Cross-dispersion [pix]")
axs[-1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]");

Each line is a monochromatic image of the slit, so it runs the full height of the frame, and
there is nothing to trace: the science frame's notions of a source and a background do not
apply to an arc. The three lamps divide the detector between them. HgAr holds the blue end,
with its brightest line near column 87, and a cluster between columns 400 and 650; Ne packs a
dense forest of lines between columns 170 and 500; Xe takes over beyond column 640, where the
other two are sparse. Look closely at any line and it is not vertical. It leans and curves
across the slit by an amount that is plain on a frame only 372 rows tall. That is the property
this chapter closes on, and the reason the next chapter exists.

## A different slit

The one difference in that table with real consequences is `MASKNAME`. The arcs went through
`LongSlits1.23Arcsec`; the science exposure through `LongSlits40.0Arcsec`, a slit **32 times
wider**.

That is deliberate, and it is the standard setup for transmission spectroscopy. A
monochromatic line is an image of the slit, so a *narrow* slit produces a sharp line whose
centroid can be measured precisely, which is exactly what a wavelength reference needs. The
science exposure has the opposite requirement: a wide slit guarantees that none of the
target's light is clipped by the slit jaws as seeing and guiding wander, at the cost of
spectral resolution. You want sharp arcs and lossless science, and those call for different
slits.

The cost shows up later, when the sky emission lines in the science frame turn out to be broad
because they fill that 40-arcsecond slit, and Chapter 7 pays it.

## Collapsing an arc frame to one dimension

To look at what each lamp emits we need a 1D cut. A plain sum over a few rows is enough here,
and it is worth being explicit about why we are *not* using `specreduce`'s extraction
machinery: that is Chapter 8's subject, it needs a background subtraction we have not done,
and an arc frame has no source to extract, since the lines fill the slit. A row sum is the honest
operation.

The band has to be **narrow**, though, and the reason is the subject of the next section.

In [ ]:
ROW, HALF = 135, 2


def row_cut(frame, row=ROW, half=HALF):
    """Sum a narrow band of rows into a 1D cut along the dispersion axis."""
    return frame.data[row - half : row + half + 1].sum(axis=0)


cuts = [row_cut(frame) for frame in arcs]

print(
    f"{'lamp':>5}  {'exptime':>7}  {'brightest line':>14}  {'per second':>13}  {'baseline':>10}"
)
for lamp, cut in zip(lamps, cuts):
    exptime = pf.getheader(DATA_DIR / files[lamp])["EXPTIME"]
    print(
        f"{lamp:>5}  {exptime:6.1f}s  {cut.max():13.0f} e-  "
        f"{cut.max() / exptime:12.0f} e-  {np.median(cut):9.0f} e-"
    )

In [ ]:
fig, axs = plt.subplots(3, figsize=(8, 5.0), sharex="all", constrained_layout=True)

for ax, lamp, cut in zip(axs, lamps, cuts):
    ax.plot(cut - np.median(cut), lw=0.6)
    ax.text(0.01, 0.9, lamp, transform=ax.transAxes, va="top", size=8)
    ax.set(yscale="log", ylim=(30, 3e6), ylabel="Signal [e$^-$]")

axs[-1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]");

The three lamps are complementary: Ne is densest where HgAr is sparse, and Xe takes over in
the red where both thin out. HgAr on its own is a different matter. Its left-hand third holds
one very bright line near column 87 and a pair of weaker lines near column 160, at about 15
percent of it, and then nothing until column 405. The line at 87 carries a large share of the
lamp's total flux, which is why HgAr looks blue-heavy on a linear scale even though it has
almost no *lines* there.

The emptiness of the bluest columns is as much a property of the instrument as of the lamp (the
R1000R grism's blue end is the faint end of its throughput), and it has a direct consequence
three chapters later: a wavelength solution fitted to HgAr alone is anchored by one line at the
blue extreme and a crowd of lines from the middle rightwards. Chapter 10 does exactly that, with
nine usable HgAr anchors and a gap between pixels 87 and 405, before bringing in the other two
lamps to fill it.

How wide are the lines? Measuring the full width at half maximum of the strongest few lines in
each cut, by interpolating the half-maximum crossings on either side of the peak, gives about
**two and a half pixels**, consistently across the three lamps. Do not read more precision into
that than it carries: a line this narrow is barely sampled, and estimators
disagree at the level of a few tenths of a pixel; fitting a Gaussian to these same lines
returns closer to 2.1. What is solid is the *scale*, and the scale is what gets used:

- It is the window the line finder needs in Chapter 9, which refits each detection with a
  Gaussian in a window sized by an `fwhm` argument.
- It is the same window the tilt fit needs in Chapter 6.
- It is the yardstick for the next section. A distortion that moves a line by a fraction of a
  line width is negligible; one that moves it by several line widths is not.

Both of those chapters pass `fwhm=2.5`, and this is where that number comes from.

Worth noticing in passing: HgAr's five strongest lines are at columns 87, 533, 550, 622 and 803.
Chapter 10 picks its wavelength-solution anchors by hand and arrives at very nearly the same
list. The brightest lines are the easiest to centroid, so they are also the ones worth
anchoring a fit on.

## The lines are not vertical

Every operation in the book so far has assumed that **a column of pixels is a single
wavelength**. On these frames that is false, and the arc frames are where it is easiest to see,
because an arc line *is* a single wavelength by construction.

Take one window of the Ne frame and cut it at three different heights on the slit.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)

window = slice(395, 470)
columns = np.arange(nx)[window]

for row, style in zip((20, 135, 310), ("-", "--", ":")):
    cut = row_cut(arcs[1], row=row)
    ax.plot(
        columns,
        cut[window] / cut[window].max(),
        style,
        lw=1.1,
        label=f"row {row}",
    )

ax.legend(title="cross-dispersion")
ax.set(
    xlabel="Dispersion [pix]",
    ylabel="Normalized signal",
    xlim=(columns[0], columns[-1]),
);

The same physical lines, at three heights on the same slit, sit in different columns. The shift
between the top and bottom cuts is several pixels, against a line width of 2.5 pixels, so the
same wavelength has moved by more than twice the width of the feature it is being measured in.

Nothing that assumes a constant wavelength down a column survives that. A wavelength solution
fitted from one row and applied to another is blurred by it, and, less obviously but more
damagingly, sky subtraction is corrupted by it, because the sky rows subtracted from the
target's row sample slightly different wavelengths than the target's row does. Both operations
have to wait until the distortion has been measured and removed, which is why tilt correction
comes next, before any measurement is made from these frames.

The distortion is a property of this instrument, not of long-slit spectroscopy in general. It
is optical in origin: a spectrograph images the slit through a dispersing element onto a flat
detector, and the monochromatic image of the slit comes out slightly curved and tilted rather
than parallel to the detector columns, by an amount that depends entirely on the optical
design. A well-aligned spectrograph may show a fraction of a pixel across the whole slit and
need no correction at all. Chapter 6 measures the distortion on these frames, and shows how to
decide whether yours needs it.

## Summary

- **An arc lamp is a ruler**, not a measurement. Its lines are monochromatic images of the
  slit at laboratory-known wavelengths, filling the slit from edge to edge, and they are the
  reference this book uses to put physical units on the dispersion axis. Sky lines, telluric
  bands and etalons can play the same role; `specreduce` only needs lines and their wavelengths.
- **Three lamps, because one is not enough.** Ne is densest where HgAr is sparse and Xe takes
  over in the red. HgAr alone has one bright line near column 87 and nothing between columns
  87 and 405, which is why Chapter 10 starts from a single blue anchor and later brings in the
  other two lamps.
- **The arcs use a narrower slit than the science frame**, 1.23 against 40 arcseconds, so
  that their lines are sharp. Read `MASKNAME` rather than assuming; the mismatch is deliberate
  and standard, not an error in the dataset.
- **The arcs are from a different night**, 1.6 days after the science frame. Any flexure
  between the two is inherited by the wavelength solution, and nothing in the dataset can check
  it.
- **Arc lines here are about 2.5 pixels FWHM.** This sets the `fwhm` window for line finding
  in Chapter 9 and for the tilt fit in Chapter 6, and it is the yardstick against which any
  distortion is judged.
- **The lines lean.** Wavelength varies along the cross-dispersion axis by several pixels, and
  every later step assumes it does not. How much a spectrograph leans is a property of its
  optics; Chapter 6 measures it here and shows how to decide whether yours needs correcting.

Next: [Tilt Correction](06-tilt-correction.ipynb), which measures that lean, models it, and
removes it.
